# [STARTER] Exercise - Building a Multi-Step State Machine Agent

In this exercise, you will build an agent that manages a multi-step workflow using a state machine. You’ll define a custom state schema, implement step logic, connect steps (including conditional routing and loops), and run the workflow to process user input through several transformations.

## Challenge

You have learned how to use a state machine to manage workflow steps and transitions. Now, your challenge is to:

- Define a state schema with multiple fields (e.g., user_query, instructions, messages, current_tool_calls).
- Implement at least three step functions:
    - Prepare Messages: Assemble the conversation history and any required context for the LLM.
    - LLM: Call the language model to generate a response or tool call.
    - Tools: Execute any required tool calls and update the state with results.
- Connect steps to form a workflow, including:
    - Entrypoint and Termination steps to start and end the workflow.
    - Conditional routing: If the LLM response includes tool calls, route to the Tools step; otherwise, proceed to Termination.
    - Looping: After executing tools, return to the LLM step to continue the workflow until there are no more tool calls.
- Run your state machine with a sample input and inspect the state transitions and snapshots to understand how your agent processes a task step by step.


## Setup
First, let's import the necessary libraries:

In [28]:
from typing import TypedDict, List, Optional, Union
import json
from dotenv import load_dotenv

from lib.state_machine import StateMachine, Step, EntryPoint, Termination, Run
from lib.llm import LLM
from lib.messages import AIMessage, UserMessage, SystemMessage, ToolMessage
from lib.tooling import Tool, ToolCall, tool

In [29]:
load_dotenv()

False

## Define a State Schema

Create a TypedDict to represent the agent’s state, including fields for the user query, instructions, message history, and any pending tool calls.

In [30]:
class AgentState(TypedDict):
    user_query: str
    instructions: str
    messages: List[Union[AIMessage, UserMessage, SystemMessage, ToolMessage]]
    current_tool_calls: Optional[List[ToolCall]]  # None once tool_step has run them

## Define the Tools you will use

Feel free to modify to add any tool you want

In [31]:
@tool
def get_games(num_games:int=1, top:bool=True) -> List[dict]:
    """
    Returns the top or bottom N games with highest or lowest scores.    
    args:
        num_games (int): Number of games to return (default is 1)
        top (bool): If True, return top games, otherwise return bottom (default is True)
    """
    data = [
        {"Game": "The Legend of Zelda: Breath of the Wild", "Platform": "Switch", "Score": 98},
        {"Game": "Super Mario Odyssey", "Platform": "Switch", "Score": 97},
        {"Game": "Metroid Prime", "Platform": "GameCube", "Score": 97},
        {"Game": "Super Smash Bros. Brawl", "Platform": "Wii", "Score": 93},
        {"Game": "Mario Kart 8 Deluxe", "Platform": "Switch", "Score": 92},
        {"Game": "Fire Emblem: Awakening", "Platform": "3DS", "Score": 92},
        {"Game": "Donkey Kong Country Returns", "Platform": "Wii", "Score": 87},
        {"Game": "Luigi's Mansion 3", "Platform": "Switch", "Score": 86},
        {"Game": "Pikmin 3", "Platform": "Wii U", "Score": 85},
        {"Game": "Animal Crossing: New Leaf", "Platform": "3DS", "Score": 88}
    ]
    # Sort the games list by Score
    # If top is True, descending order
    sorted_games = sorted(data, key=lambda x: x['Score'], reverse=top)
    
    # Return the N games
    return sorted_games[:num_games]

@tool
def count_games(games: List[dict]) -> int:
    """
    Returns the count of games in the list.
    args:
        games (List[dict]): List of game dictionaries
    """
    return len(games)

@tool
def get_game_titles(games: List[dict]) -> List[str]:
    """
    Returns the titles of the games in the list.
    args:
        games (List[dict]): List of game dictionaries
    """
    return [game['Game'] for game in games]

@tool   
def get_game_platforms(games: List[dict]) -> List[str]:
    """
    Returns the platforms of the games in the list.
    args:
        games (List[dict]): List of game dictionaries
    """
    return [game['Platform'] for game in games]

@tool
def get_game_scores(games: List[dict]) -> List[int]:
    """
    Returns the scores of the games in the list.
    args:
        games (List[dict]): List of game dictionaries
    """
    return [game['Score'] for game in games]

@tool
def get_game_info(games: List[dict]) -> List[str]:
    """
    Returns the game information in a formatted string.
    args:
        games (List[dict]): List of game dictionaries
    """
    return [f"{game['Game']} ({game['Platform']}): {game['Score']}" for game in games]


In [32]:
tools = [get_games, count_games, get_game_titles, get_game_platforms, get_game_scores, get_game_info]

## Create the Steps

Write functions for each step in your workflow:


**Prepare Messages**: Build the message list for the LLM.

In [ ]:
def prepare_messages_step(state: AgentState) -> AgentState:
    """
    Prepares the messages for the agent based on the current state.
    
    Args:
        state (AgentState): The current state of the agent.
    
    Returns:
        AgentState: The agent state with the prepared messages.
    """
    # Create a SystemMessage using the instructions
    system_message = SystemMessage(content=state["instructions"])
    
    # Create a UserMessage using the user_query
    user_message = UserMessage(content=state["user_query"])
    
    # Return the agent state with the messages list containing the SystemMessage and UserMessage
    return {
        "messages": [system_message, user_message],
        "instructions": state["instructions"],
        "user_query": state["user_query"],
        "current_tool_calls": state.get("current_tool_calls", [])
    }

**LLM Step**: Call the language model and check for tool calls.

In [34]:
def llm_step(state: AgentState) -> AgentState:
    """
    Processes the state through an LLM and handles tool calls.
    
    Args:
        state (AgentState): The current state of the agent.
    Returns:
        AgentState: The agent state with updated messages and current tool calls.
    """
    # Initialize the LLM with the tools
    llm = LLM(tools=tools)
    
    # Invoke the LLM with the prepared messages
    response = llm.invoke(state["messages"])
    
    # Append the AIMessage to the messages list. Keep its tool_calls: the API
    # rejects tool results unless the preceding assistant message requested them
    ai_message = AIMessage(content=response.content, tool_calls=response.tool_calls)
    updated_messages = state["messages"] + [ai_message]
    
    # Update the current_tool_calls with the tool calls from the response
    updated_tool_calls = response.tool_calls
    
    # Return the updated AgentState
    return {
        "messages": updated_messages,
        "instructions": state["instructions"],
        "user_query": state["user_query"],
        "current_tool_calls": updated_tool_calls or [],
    }

**Tool Step**: Execute any tool calls and update the state.

In [35]:
def tool_step(state: AgentState) -> AgentState:
    """
    Executes any pending tool calls and updates the state.
    
    Args:
        state (AgentState): The current state of the agent.
    
    Returns:
        AgentState: The updated agent state.
    """
    # Get the current tool calls (the key may be missing or set to None)
    tool_calls = state.get("current_tool_calls") or []
    
    # Initialize the updated messages list
    updated_messages = state["messages"][:]
    
    # Execute each tool call and create a ToolMessage for each
    available_tools = {tool.name: tool for tool in tools}

    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        tool_arguments = json.loads(tool_call.function.arguments)
        selected_tool = available_tools[tool_name]
        tool_result = selected_tool(**tool_arguments)

        # Create a ToolMessage with the result; the API needs tool_call_id to
        # match the result to the call the model made
        tool_message = ToolMessage(
            content=json.dumps(tool_result),
            tool_call_id=tool_call.id,
            name=tool_name,
        )
        
        # Add the ToolMessage to the updated messages list
        updated_messages.append(tool_message)
    
    # Return the updated AgentState with the new messages and no current tool calls
    return {
        "messages": updated_messages,
        "instructions": state["instructions"],
        "user_query": state["user_query"],
        "current_tool_calls": None,
    }

## Build and Connect the State Machine

Add your steps to the state machine, and connect them with transitions. Use conditional routing to decide whether to call tools or terminate, and loop as needed.

In [36]:
workflow = StateMachine(AgentState)

In [ ]:
entry = EntryPoint[AgentState]()
message_prep = Step[AgentState]("message_prep", prepare_messages_step)
llm_processor = Step[AgentState]("llm_processor", llm_step)
tool_executor = Step[AgentState]("tool_executor", tool_step)
termination = Termination[AgentState]()

workflow.add_steps([
    entry,
    message_prep,
    llm_processor,
    tool_executor,
    termination,
])

In [38]:
# Add transitions
workflow.connect(entry, message_prep)
workflow.connect(message_prep, llm_processor)

# Transition based on whether there are tool calls
def check_tool_calls(state: AgentState) -> Union[Step[AgentState], str]:
    """Transition logic: Check if there are tool calls"""
    if state.get("current_tool_calls"):
        return tool_executor
    return termination

# Routing: If tool calls -> tool_executor
workflow.connect(
    source=llm_processor, 
    targets=[tool_executor, termination], 
    condition=check_tool_calls
)

# Looping: Go back to llm after tool execution
workflow.connect(
    source=tool_executor, 
    targets=llm_processor
)

## Run the Workflow

In [39]:
initial_state: AgentState = {
    "user_query": "What's the best game in the dataset?",
    "instructions": "You can bring insights about a game dataset based on users questions",
    "messages": [],
    "current_tool_calls": None,
}

In [40]:
run_object = workflow.run(initial_state)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__


In [41]:
final_state = run_object.get_final_state()

if final_state is None:
    raise RuntimeError("The workflow did not produce a final state.")

final_state["messages"]

[SystemMessage(content='You can bring insights about a game dataset based on users questions', role='system'),
 UserMessage(content="What's the best game in the dataset?", role='user'),
 AIMessage(content='', role='assistant', tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_uepH1q6wnthRGKWVKFEtVtwa', function=Function(arguments='{"num_games":1,"top":true}', name='get_games'), type='function')]),
 ToolMessage(content='[{"Game": "The Legend of Zelda: Breath of the Wild", "Platform": "Switch", "Score": 98}]', role='tool', tool_call_id='call_uepH1q6wnthRGKWVKFEtVtwa', name='get_games'),
 AIMessage(content='The best game in the dataset is **The Legend of Zelda: Breath of the Wild** with a score of **98** on the **Switch** platform.', role='assistant', tool_calls=None)]

In [ ]:
initial_state_2: AgentState = {
    "user_query": "What are the top 3 games and their platforms?",
    "instructions": "You can bring insights about a game dataset based on users questions",
    "messages": [],
    "current_tool_calls": None,
}

run_object_2 = workflow.run(initial_state_2)
final_state_2 = run_object_2.get_final_state()

if final_state_2 is None:
    raise RuntimeError("The workflow did not produce a final state.")

final_state_2["messages"]

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__


[SystemMessage(content='You can bring insights about a game dataset based on users questions', role='system'),
 UserMessage(content='What are the top 3 games and their platforms?', role='user'),
 AIMessage(content='', role='assistant', tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_jTRz1LxKT8JDP8VIWoYrnhl6', function=Function(arguments='{"num_games": 3, "top": true}', name='get_games'), type='function'), ChatCompletionMessageFunctionToolCall(id='call_aEGoMwoKi7A7AoBIdUxdbvHG', function=Function(arguments='{"num_games": 3, "top": true}', name='get_games'), type='function')]),
 ToolMessage(content='[{"Game": "The Legend of Zelda: Breath of the Wild", "Platform": "Switch", "Score": 98}, {"Game": "Super Mario Odyssey", "Platform": "Switch", "Score": 97}, {"Game": "Metroid Prime", "Platform": "GameCube", "Score": 97}]', role='tool', tool_call_id='call_jTRz1LxKT8JDP8VIWoYrnhl6', name='get_games'),
 ToolMessage(content='[{"Game": "The Legend of Zelda: Breath of the Wild", "Platfor

## Optional 

Create an Agent class to encapsulate State Machine logic. Then try adding more tools, and experiment with different user queries to see how the workflow adapts.

In [ ]:
class Agent:
    def __init__(self, 
                 model_name: str,
                 instructions: str, 
                 tools: List[Tool] = [],
                 temperature: float = 0.7):
        """
        Initialize an Agent instance
        
        Args:
            model_name: Name/identifier of the LLM model to use
            instructions: System instructions for the agent
            tools: Optional list of tools available to the agent
            temperature: Temperature parameter for LLM (default: 0.7)
        """
        self.instructions = instructions
        self.tools = tools if tools else []
        self.model_name = model_name
        self.temperature = temperature
                
        # Initialize state machine
        self.workflow = self._create_state_machine()

    def _prepare_messages_step(self, state: AgentState) -> AgentState:
        return prepare_messages_step(state)

    def _llm_step(self, state: AgentState) -> AgentState:
        return llm_step(state)

    def _tool_step(self, state: AgentState) -> AgentState:
        return tool_step(state)

    def _create_state_machine(self) -> StateMachine[AgentState]:
        # create a state machine for the agent with the defined steps and transitions
        workflow = StateMachine(AgentState)

        entry = EntryPoint[AgentState]()
        message_prep = Step[AgentState](
            "message_prep",
            self._prepare_messages_step,
        )
        llm = Step[AgentState]("llm", self._llm_step)
        tool_executor = Step[AgentState]("tool", self._tool_step)
        termination = Termination[AgentState]()

        workflow.add_steps([
            entry,
            message_prep,
            llm,
            tool_executor,
            termination,
        ])

        workflow.connect(entry, message_prep)
        workflow.connect(message_prep, llm)

        def route_after_llm(
            state: AgentState,
        ) -> Union[Step[AgentState], str]:
            if state.get("current_tool_calls"):
                return tool_executor
            return termination

        workflow.connect(
            source=llm,
            targets=[tool_executor, termination],
            condition=route_after_llm,
        )
        workflow.connect(tool_executor, llm)

        return workflow

    def invoke(self, query: str) -> Run:
        """
        Run the agent on a query
        
        Args:
            query: The user's query to process
            
        Returns:
            The final run object after processing
        """

        initial_state: AgentState = {
            "user_query": query,
            "instructions": self.instructions,
            "messages": [],
            "current_tool_calls": None,
        }

        run_object = self.workflow.run(initial_state)

        return run_object
